HW6 Analysis
How many principal components were required to retain 95% of the total variance compared to the original 4,096 dimensions? Based on your reconstruction plots in Task 3, what visual details are lost at lower values of k; e.g., k=10, and at what point does the reconstructed image become visually indistinguishable from the original?

  1. It took 123 components to keep 95% of the variance, which is only 3.0% of the 4,096 pixel columns (for 90% it was 66 and for 99% it was 260). This works because neighboring pixels in a face are strongly related to each other, so most of the columns repeat the same information. When PCA was fitted on the 320 training faces only, 95% needed 111 components, which is a little different because it saw fewer faces. At k=10 the reconstructions are blurry blobs that show the head shape, the hair line and where the image is light or dark, but the eyes are soft, the mouth is smeared, the glasses and the beard texture are gone, and the faces look much more alike than the originals. The MSE for the 5 faces was 0.0043 at k=10 and fell to 0.0018 at k=50, 0.0010 at k=100, 0.0004 at k=200 and 0.0001 at k=300. At k=50 the glasses, eyes and smile are clearly visible and I could tell who it was, but the teeth and skin texture are still soft. At k=100 and k=200 only very small details were missing, and at k=300 I could not see a difference from the original, even though the MSE was still not exactly 0.


What do the earliest principal components (the first few "Eigenfaces") physically capture in the images? Why do later principal components appear increasingly like fine-grained pixel noise?

  2. The first eigenfaces look like faint faces and capture the biggest differences between the 400 pictures. PC 1 (23.81% of the variance) and PC 2 (13.99%) are mostly the overall brightness of the face compared to the background, so they are about lighting and the outline of the head and hair. After that the components start to pick out parts of the face, like the eyes and eyebrows, the mouth and smile (PC 5 and PC 6) and the glasses (PC 7). The first 15 components together explain 72.12% of the variance, but the 15th only explains 1.06% and the 100th only 0.08%. Each component has to be perpendicular to all of the ones before it, so the later components can only describe what is left over, which is small differences between single pixels and small local patterns that change from picture to picture. These carry very little variance and do not look like anything we can recognize, so they look like static. This is also why throwing them away at 95% removes noise but keeps the face.


Why is it critical to fit the PCA transformer strictly on the training split rather than on the entire dataset prior to splitting? What negative impacts occur if PCA is fitted on the full dataset before training downstream classifiers?

  3. PCA learns its axes from the data it is given, so if I fit it on all 400 faces the test faces help decide where the axes point, which means information from the test set leaks into training. In this notebook I fitted PCA(n_components=0.95) on X_train only and then used transform() on X_test, the same way the scaler was fitted on the training set and applied to the test set in my earlier homeworks. The two PCAs really were different: the one fitted on all 400 faces needed 123 components for 95%, while the one fitted on the training faces needed 111. If I had fitted it on everything, the test accuracy would look better than it really is because the components were already shaped to represent the test faces, and I would be overestimating how the model does on new pictures. With only 320 training images the PCA can make at most 320 components, so the leak matters more because the 80 test faces make up a large share of what PCA sees.


Compare the classification accuracy of Logistic Regression against Kernel SVM and the Ensemble Classifier on PCA-reduced features. Did non-linear kernels or ensemble methods yield substantial accuracy gains? Explain why classical classifiers (linear, non-linear, or ensemble) hit a performance ceiling when operating directly on flattened pixel vectors or global PCA projections.

  4. Logistic Regression on the PCA features scored 96.25% (and 97.5% on the raw pixels), the SVC with an RBF kernel scored 93.75% and the soft voting ensemble scored 95.0%. These are 3, 5 and 4 misclassified faces out of 80, so the non-linear models did not give any gain, and the SVC was actually 2 faces worse than Logistic Regression. The ensemble was also below its own best single model, which is similar to my diabetes homework, where the voting models did not beat the best single model. All of the models ended up between about 94% and 97.5%, and the differences between them are only 1 or 2 faces, so I cannot say any of them is really better. All of these models see a face as 4,096 (or 111) numbers in a row and have no idea which pixels are next to each other, so they cannot see an eye, a nose or a mouth, only numbers. There are only 8 training pictures of each of the 40 people in my 80/20 split, and the same person changes with lighting, expression and glasses, which moves a lot of pixels at once. PCA also keeps the directions with the most variance, and the biggest variance here is lighting (PC 1 and PC 2), which does not necessarily separate one person from another. Because every model is getting the same pixel-based features, using a fancier model on top of them cannot add information that these features do not contain.


Based on your findings in Task 8 regarding accuracy and training efficiency, why are global linear transformations like PCA and classical pixel classifiers insufficient for complex computer vision tasks?

  5. In the Task 8 plot, PCA cut the Logistic Regression training time from 6.86s to 0.19s (about 36 times faster) while the accuracy went from 97.5% to 96.25%, which is one face out of 80, so PCA was useful for speed. The SVC took 0.16s and the soft voting ensemble took 0.62s, the longest of the PCA models, because it trains two SVCs (with probability=True) and 100 trees, and it did not give a better accuracy than the plain Logistic Regression. So I got faster training from PCA but no accuracy gain from the heavier models, which points to the features as the limit and not the classifier. PCA is global and linear, so one set of axes is applied to the whole image and it cannot find local parts like an eye, and if the face moves, tilts or is lit from a different side, the pixel values change completely even though it is the same person. The Olivetti pictures are small, centered, all taken against a dark background and very similar to each other, which is the easiest possible case for this approach, so real photos with different backgrounds, sizes and angles would be much harder. This is why I think methods that learn local features directly from the image, like a convolutional network, are needed for harder computer vision tasks.


